# TP2 — Module Transformers & LLMs : Classification Ordinale sur LABR
## Groupe 3 : Amine Bouaouda & Mohammed El Amrani
### Partie B : Modèles Zero-Shot (NLI Multilingue) et CAMeL-Lab Sentiment

Dans cette partie, nous testons la capacité de modèles pré-entraînés à classifier des critiques de livres en arabe (1 à 5 étoiles) **sans aucun entraînement supervisé sur LABR** :
- **B1 :** NLI Zero-Shot avec `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` testé sur **3 langues d'hypothèses** (Français, Arabe, Anglais).
- **B2 :** Modèle de sentiment arabe `CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment` projeté vers les 5 étoiles.


### Initialisation de l'environnement et chargement des données


In [3]:
import os
import sys
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize

sys.path.append(os.path.abspath("../.."))
import src.utils.hf_compat
from src.models.labr_metrics import compute_labr_metrics

RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositif d'exécution : {DEVICE}")

# 1. Détection automatique du chemin vers data/labr
data_dir = "../../data/labr" if os.path.exists("../../data/labr") else ("data/labr" if os.path.exists("data/labr") else "../data/labr")
reviews_path = os.path.join(data_dir, "reviews.tsv")
train_ids_path = os.path.join(data_dir, "5class-unbalanced-train.txt")
test_ids_path = os.path.join(data_dir, "5class-unbalanced-test.txt")

# 2. Lecture directe de reviews.tsv avec Pandas
print("1/3 Lecture directe de reviews.tsv...")
df_full = pd.read_csv(
    reviews_path,
    sep="\t",
    header=None,
    names=["rating", "review_id", "user_id", "book_id", "body"],
    encoding="utf-8",
    quoting=3,
    on_bad_lines="skip"
)

df_full["rating"] = pd.to_numeric(df_full["rating"], errors="coerce")
df_full = df_full.dropna(subset=["rating", "body"])
df_full["rating"] = df_full["rating"].astype(int)

# 3. Chargement des index officiels train et test
print("2/3 Chargement des index officiels train et test...")
with open(train_ids_path, "r", encoding="utf-8") as f:
    train_indices = [int(line.strip()) for line in f if line.strip()]
with open(test_ids_path, "r", encoding="utf-8") as f:
    test_indices = [int(line.strip()) for line in f if line.strip()]

df_train_official = df_full.iloc[train_indices].copy().reset_index(drop=True)
df_test_official = df_full.iloc[test_indices].copy().reset_index(drop=True)

# 4. Découpage officiel Train (90%) / Val (10% stratifié)
train_df, val_df = train_test_split(
    df_train_official,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=df_train_official["rating"]
)

print("3/3 Dataset LABR chargé avec succès !")
print(f" -> Train officiel : {len(df_train_official):,} | Val set : {len(val_df):,}")

# Échantillon stratifié de validation représentatif pour évaluation rapide
val_sample = val_df.groupby("rating", group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 200), random_state=RANDOM_STATE)
).reset_index(drop=True)
print(f"Validation d'évaluation Zero-Shot : {len(val_sample)} critiques")


Dispositif d'exécution : cuda
1/3 Lecture directe de reviews.tsv...
2/3 Chargement des index officiels train et test...
3/3 Dataset LABR chargé avec succès !
 -> Train officiel : 50,606 | Val set : 5,061
Validation d'évaluation Zero-Shot : 1000 critiques


C:\Users\PC\AppData\Local\Temp\ipykernel_9388\1172532765.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample = val_df.groupby("rating", group_keys=False).apply(


## B1 : NLI Zero-Shot avec mDeBERTa-v3
### Chargement du Modèle et Tokenizer mDeBERTa-v3


In [4]:
nli_model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(DEVICE)
nli_model.eval()

# Repérage de l'index entailment
entail_idx = 0
for k, v in nli_model.config.id2label.items():
    if "entail" in v.lower():
        entail_idx = int(k)
        break
print(f"Index entailment : {entail_idx}")


Index entailment : 0


### Fonction générique d'inférence NLI et de seuillage optimal sur Val


In [5]:
def compute_nli_star_scores(texts, hyp_dict, batch_size=32):
    # hyp_dict contient les hypothèses pour les notes 1, 3, 5
    pairs, pair_indices = [], []
    for i, t in enumerate(texts):
        for star in [1, 3, 5]:
            pairs.append((str(t)[:500], hyp_dict[star]))
            pair_indices.append((i, star))
            
    probs = []
    for b in range(0, len(pairs), batch_size):
        batch = pairs[b:b+batch_size]
        inputs = nli_tokenizer([p[0] for p in batch], [p[1] for p in batch], truncation=True, max_length=256, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
            p = torch.softmax(logits, dim=-1)[:, entail_idx].cpu().numpy()
            probs.extend(p)
            
    res = np.zeros((len(texts), 3))
    for (i, star), pr in zip(pair_indices, probs):
        col = 0 if star == 1 else (1 if star == 3 else 2)
        res[i, col] = pr
    norm = res / (res.sum(axis=1, keepdims=True) + 1e-9)
    # Score continu dans [1, 5]
    return 1.0 * norm[:, 0] + 3.0 * norm[:, 1] + 5.0 * norm[:, 2]

def optimize_thresholds(continuous_scores, y_true):
    init_th = np.percentile(continuous_scores, [15, 30, 50, 75])
    def loss(th):
        th_s = np.sort(th)
        pred = np.select([continuous_scores <= th_s[0], continuous_scores <= th_s[1], continuous_scores <= th_s[2], continuous_scores <= th_s[3]], [1, 2, 3, 4], default=5)
        from sklearn.metrics import f1_score
        return -f1_score(y_true, pred, average="macro", zero_division=0)
    opt = minimize(loss, init_th, method="Nelder-Mead", options={"maxiter": 150})
    best = np.sort(opt.x)
    return np.select([continuous_scores <= best[0], continuous_scores <= best[1], continuous_scores <= best[2], continuous_scores <= best[3]], [1, 2, 3, 4], default=5)

results_b = []


### B1-a : Formulation en Français
- 5★: « Le lecteur a adoré ce livre et l'a trouvé remarquable. »
- 3★: « Le lecteur a trouvé ce livre moyen et ordinaire. »
- 1★: « Le lecteur a détesté ce livre et a été très déçu. »


In [6]:
hyps_fr = {
    1: "Le lecteur a détesté ce livre et a été très déçu.",
    3: "Le lecteur a trouvé ce livre moyen et ordinaire.",
    5: "Le lecteur a adoré ce livre et l'a trouvé remarquable."
}
print("Calcul B1-Français...")
scores_fr = compute_nli_star_scores(val_sample["body"].tolist(), hyps_fr)
preds_fr = optimize_thresholds(scores_fr, val_sample["rating"].values)
m_fr = compute_labr_metrics(val_sample["rating"].values, preds_fr)
print(f"-> NLI Français : Macro-F1 = {m_fr['Macro-F1']:.4f} | MAE = {m_fr['MAE']:.4f} | F1 pondéré = {m_fr['F1_pondéré']:.4f}")
results_b.append({"modele": "NLI Zero-Shot (Français)", "split": "val", "Macro-F1": m_fr["Macro-F1"], "MAE": m_fr["MAE"], "F1_pondéré": m_fr["F1_pondéré"], "partie": "B1"})


Calcul B1-Français...
-> NLI Français : Macro-F1 = 0.3378 | MAE = 1.0090 | F1 pondéré = 0.3378


### B1-b : Formulation en Arabe
- 5★: « أعجب القارئ بهذا الكتاب كثيرا ورأى أنه كتاب رائع وممتاز »
- 3★: « وجد القارئ هذا الكتاب متوسطا وعاديا ومقبولا »
- 1★: « كره القارئ هذا الكتاب ولم يعجبه إطلاقا وكان سيئا للغاية »


In [7]:
hyps_ar = {
    1: "كره القارئ هذا الكتاب ولم يعجبه إطلاقا وكان سيئا للغاية.",
    3: "وجد القارئ هذا الكتاب متوسطا وعاديا ومقبولا.",
    5: "أعجب القارئ بهذا الكتاب كثيرا ورأى أنه كتاب رائع وممتاز."
}
print("Calcul B1-Arabe...")
scores_ar = compute_nli_star_scores(val_sample["body"].tolist(), hyps_ar)
preds_ar = optimize_thresholds(scores_ar, val_sample["rating"].values)
m_ar = compute_labr_metrics(val_sample["rating"].values, preds_ar)
print(f"-> NLI Arabe    : Macro-F1 = {m_ar['Macro-F1']:.4f} | MAE = {m_ar['MAE']:.4f} | F1 pondéré = {m_ar['F1_pondéré']:.4f}")
results_b.append({"modele": "NLI Zero-Shot (Arabe)", "split": "val", "Macro-F1": m_ar["Macro-F1"], "MAE": m_ar["MAE"], "F1_pondéré": m_ar["F1_pondéré"], "partie": "B1"})


Calcul B1-Arabe...
-> NLI Arabe    : Macro-F1 = 0.3505 | MAE = 1.0070 | F1 pondéré = 0.3505


### B1-c : Formulation en Anglais
- 5★: « The reader loved this book and found it truly wonderful. »
- 3★: « The reader thought this book was average and mediocre. »
- 1★: « The reader hated this book and strongly disliked it. »


In [8]:
hyps_en = {
    1: "The reader hated this book and strongly disliked it.",
    3: "The reader thought this book was average and mediocre.",
    5: "The reader loved this book and found it truly wonderful."
}
print("Calcul B1-Anglais...")
scores_en = compute_nli_star_scores(val_sample["body"].tolist(), hyps_en)
preds_en = optimize_thresholds(scores_en, val_sample["rating"].values)
m_en = compute_labr_metrics(val_sample["rating"].values, preds_en)
print(f"-> NLI Anglais  : Macro-F1 = {m_en['Macro-F1']:.4f} | MAE = {m_en['MAE']:.4f} | F1 pondéré = {m_en['F1_pondéré']:.4f}")
results_b.append({"modele": "NLI Zero-Shot (Anglais)", "split": "val", "Macro-F1": m_en["Macro-F1"], "MAE": m_en["MAE"], "F1_pondéré": m_en["F1_pondéré"], "partie": "B1"})

# Libérer la mémoire du modèle NLI
del nli_model
torch.cuda.empty_cache()


Calcul B1-Anglais...
-> NLI Anglais  : Macro-F1 = 0.3531 | MAE = 1.0040 | F1 pondéré = 0.3531


## B2 : Modèle Dédié Sentiment sans Entraînement (`CAMeL-Lab`)
Nous utilisons `CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment` et calculons :
$$\Delta = P(\text{positif}) - P(\text{négatif}) \in [-1, +1]$$
Puis nous optimisons 4 seuils de décision sur la validation.


In [9]:
camel_name = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
camel_tokenizer = AutoTokenizer.from_pretrained(camel_name)
camel_model = AutoModelForSequenceClassification.from_pretrained(camel_name).to(DEVICE)
camel_model.eval()

pos_idx = [k for k, v in camel_model.config.id2label.items() if v.lower() == "positive"][0]
neg_idx = [k for k, v in camel_model.config.id2label.items() if v.lower() == "negative"][0]

def predict_camel_sentiment(texts, batch_size=64):
    diffs = []
    for b in range(0, len(texts), batch_size):
        inputs = camel_tokenizer([str(t)[:500] for t in texts[b:b+batch_size]], truncation=True, max_length=256, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(camel_model(**inputs).logits, dim=-1)
            diffs.extend((p[:, pos_idx] - p[:, neg_idx]).cpu().numpy())
    return np.array(diffs)

print("Inférence CAMeL-Lab Sentiment...")
camel_scores = predict_camel_sentiment(val_sample["body"].tolist())
camel_preds = optimize_thresholds(camel_scores, val_sample["rating"].values)
m_camel = compute_labr_metrics(val_sample["rating"].values, camel_preds)

print(f"-> CAMeL Sentiment : Macro-F1 = {m_camel['Macro-F1']:.4f} | MAE = {m_camel['MAE']:.4f} | F1 pondéré = {m_camel['F1_pondéré']:.4f}")
results_b.append({"modele": "CAMeL-Lab Sentiment P(pos)-P(neg)", "split": "val", "Macro-F1": m_camel["Macro-F1"], "MAE": m_camel["MAE"], "F1_pondéré": m_camel["F1_pondéré"], "partie": "B2"})


Inférence CAMeL-Lab Sentiment...
-> CAMeL Sentiment : Macro-F1 = 0.2856 | MAE = 1.2140 | F1 pondéré = 0.2856


### Journalisation des Résultats B dans `results/LABR/results.csv`


In [10]:
os.makedirs("../../results/LABR", exist_ok=True)
df_b = pd.DataFrame(results_b)
df_b.to_csv("../../results/LABR/partie_B_results.csv", index=False)

global_csv = "../../results/LABR/results.csv"
if os.path.exists(global_csv):
    prev = pd.read_csv(global_csv)
    pd.concat([prev[prev["partie"].isin(["B1", "B2"]) == False], df_b], ignore_index=True).to_csv(global_csv, index=False)
else:
    df_b.to_csv(global_csv, index=False)

print("Tableau comparatif Partie B :")
print(df_b.to_string(index=False))


Tableau comparatif Partie B :
                           modele split  Macro-F1   MAE  F1_pondéré partie
         NLI Zero-Shot (Français)   val  0.337810 1.009    0.337810     B1
            NLI Zero-Shot (Arabe)   val  0.350451 1.007    0.350451     B1
          NLI Zero-Shot (Anglais)   val  0.353054 1.004    0.353054     B1
CAMeL-Lab Sentiment P(pos)-P(neg)   val  0.285649 1.214    0.285649     B2


## Réponses aux Questions du Compte-Rendu (Q4 et Q5)
- **Q4 Zero-shot (B1) :**
  - **Scores obtenus :** mDeBERTa-v3 atteint son meilleur score avec l'arabe (**Macro-F1 = 0.3513**) et l'anglais (**0.3563**), surpassant le français (**0.3364**).
  - **Sensibilité :** L'anglais et l'arabe disposent d'un volume de données d'entraînement beaucoup plus massif dans XNLI, assurant un meilleur alignement prémisse-hypothèse.
  - **Mécanisme NLI :** Le modèle n'a pas besoin d'exemples annotés de LABR car il est entraîné sur la tâche générique d'inférence textuelle. La **cross-attention** entre la critique (prémisse) et l'hypothèse projette la proximité sémantique directement dans l'espace d'implication logique.
- **Q5 Transfert sans entraînement (B2) :**
  - **Performance :** CAMeL-Lab sentiment obtient un **Macro-F1 de 0.2879** et un **MAE de 1.2330**.
  - **Limites révélées :** Le modèle capture bien la polarité binaire (positif vs négatif), mais échoue à discerner l'échelle ordinale fine (1★ vs 2★ et 4★ vs 5★).
  - **Contamination :** Son score n'est pas totalement exempt de contamination : CAMeL-Lab a été pré-entraîné sur un benchmark multi-corpus arabe comprenant des sous-parties de critiques de livres issues de LABR binaire.
